# 13.5 偏好怎麼產生梯度？

# 第 13 章：把回答偏好變成訓練

前置：第7章SFT，以及第8–9章的「好回答」規則。可以直接走這條路，不必先學圖片。把兩篇答案像兩道菜一起試喫：喜歡哪一篇，與「兩篇是否都安全」是不同問題。




In [ ]:
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：試喫兩道菜：比起原廚師，更偏向好回答**

chosen 與 rejected 必須對同一個問題。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/dpo.svg")))

**先想一想：**chosen還不佔優勢時，下降方向應提升哪個分數？

DPO鼓勵相較於基準更偏向chosen，而不需要先訓練一個獨立評分網路。先用四個scalar log分數看梯度方向。

**把直覺寫成數學：**L=−log sigmoid(beta·margin)；對chosen的導數通常為負，gradient descent會提高它。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
from tiny_perceptron.alignment import dpo_loss

chosen = torch.tensor([-4.0], requires_grad=True)
rejected = torch.tensor([-3.0], requires_grad=True)
loss = dpo_loss(chosen, rejected, torch.tensor([-4.0]), torch.tensor([-3.0]), beta=0.1)
loss.backward()
print(loss.item(), chosen.grad, rejected.grad)
assert chosen.grad < 0 and rejected.grad > 0

**如何讀結果：**四個值是log probabilities；真實模型要用13.3的答案mask和固定reference。CLI --task dpo。

**只改一件事：**只交換chosen/rejected，預測梯度方向。
